# Un agente visión-lenguaje + segmentación

Notebook opcional del **módulo 23** del curso (en la app: `uv run dev` y la ruta `#/modulo/sistemas-inteligentes`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

> ⚠️ **Requiere GPU** con CUDA para ejecutarse en un tiempo razonable.

Un modelo visión-lenguaje (Qwen2.5-VL) propone un punto para la referencia del usuario y SAM lo convierte en una máscara. Es la idea del agente SAM 3 del capítulo 12, en versión mínima.

In [ ]:
import re

import matplotlib.pyplot as plt
import requests
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, SamModel, SamProcessor

url = "https://huggingface.co/ybelkada/segment-anything/resolve/main/assets/car.png"
imagen = Image.open(requests.get(url, stream=True).raw).convert("RGB")
vlm = AutoModelForImageTextToText.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype=torch.bfloat16, device_map="cuda")
vproc = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
sam = SamModel.from_pretrained("facebook/sam-vit-base").to("cuda")
sproc = SamProcessor.from_pretrained("facebook/sam-vit-base")

In [ ]:
def planificar(referencia):
    mensajes = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text":
        f"Da las coordenadas en píxeles (x, y) de un punto sobre: {referencia}. Responde solo 'x, y'."}]}]
    prompt = vproc.apply_chat_template(mensajes, add_generation_prompt=True)
    x = vproc(text=[prompt], images=[imagen], return_tensors="pt").to("cuda")
    texto = vproc.batch_decode(vlm.generate(**x, max_new_tokens=20)[:, x["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    return [int(v) for v in re.findall(r"\d+", texto)[:2]]


def segmentar(punto):
    x = sproc(imagen, input_points=[[punto]], return_tensors="pt").to("cuda")
    with torch.no_grad():
        y = sam(**x)
    m = sproc.image_processor.post_process_masks(y.pred_masks.cpu(), x["original_sizes"].cpu(), x["reshaped_input_sizes"].cpu())[0][0]
    return m[int(y.iou_scores[0, 0].argmax())]


punto = planificar("la rueda delantera del coche")
plt.imshow(imagen)
plt.imshow(segmentar(punto), alpha=0.5)
plt.scatter(*punto, c="red")
plt.show()

**Para seguir:** añade un paso de verificación (el VLM mira la máscara superpuesta y decide si es correcta) y un reintento con otro punto si no lo es.